## load


In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split , GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import(
    accuracy_score,
    confusion_matrix,
    classification_report
)

df = pd.read_csv('Titanic-Dataset.csv')

## understand

In [2]:
print(df.shape)
print(df.columns)
print(df.info())
print(df.describe())
print(df['Survived'].value_counts())

(891, 12)
Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB
None
       PassengerId    Survived      Pclass         Age       SibSp  \
count   891.000000  891.

## clean

In [3]:
print(df.isnull().sum())

df = df.drop_duplicates()

# df = df.drop(
#     ['PassengerId', 'Name' , 'Ticket' , 'Cabin'],
#     axis=1
# )

print(df.isnull().sum())

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


## analize

In [4]:
print(df.groupby('Sex')['Survived'].mean())
print(df.groupby('Pclass')['Survived'].mean())
print(df.groupby('Embarked')['Survived'].mean())

Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64
Pclass
1    0.629630
2    0.472826
3    0.242363
Name: Survived, dtype: float64
Embarked
C    0.553571
Q    0.389610
S    0.336957
Name: Survived, dtype: float64


## prepare

In [5]:
x = df.drop('Survived', axis=1)
y = df['Survived']

numeric_features = ['Age', 'SibSp', 'Parch', 'Fare', 'Pclass']
categorical_feature = ['Sex', 'Embarked']

numeric_piprline = Pipeline([
    ('imputer',
    SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer',
    SimpleImputer(strategy='most_frequent')),
    ('encoder',
    OneHotEncoder(handle_unknown='ignore'))
])

Preprocessor = ColumnTransformer([
    ('num',
    numeric_piprline, numeric_features),
    ('cat', categorical_pipeline,categorical_feature)
])

x_train , x_test, y_train , y_test = train_test_split(x,y, test_size=0.2, random_state=42, stratify=y)

### train


In [6]:
model = Pipeline([
    ('preprocessor',
Preprocessor ),

('classifier',
LogisticRegression(max_iter=1000))

])

model.fit(x_train, y_train)
prediction = model.predict(x_test)

## evaluation

In [7]:
print('Accuracy:' ,
accuracy_score(y_test, prediction))

print('\nConfusion Matrix:' ,
confusion_matrix(y_test, prediction))

print('\nClassification report:' ,
classification_report(y_test, prediction))

Accuracy: 0.8044692737430168

Confusion Matrix: [[98 12]
 [23 46]]

Classification report:               precision    recall  f1-score   support

           0       0.81      0.89      0.85       110
           1       0.79      0.67      0.72        69

    accuracy                           0.80       179
   macro avg       0.80      0.78      0.79       179
weighted avg       0.80      0.80      0.80       179



## tune

In [8]:
param_grid = {
    'classifier__C' : [0.01, 0.1 , 1 , 10 , 100] 
}

grid = GridSearchCV(
    model,
    param_grid,
    cv=5,
    scoring='accuracy'
)

grid.fit(x_train , y_train)
best_model = grid.best_estimator_

print('Best Parameters:',
grid.best_params_)

print('Best Parameters:',
grid.best_params_)

print('Best CV score:',
grid.best_score_)


Best Parameters: {'classifier__C': 0.1}
Best Parameters: {'classifier__C': 0.1}
Best CV score: 0.8020388062641584


## predict

In [9]:
final_prediction = best_model.predict(x_test)

print('Final Accuracy:',
accuracy_score(y_test, prediction))

new_passenger = pd.DataFrame({
    'Pclass': [3],
    'Sex' : ['male'],
    'Age' : [25],
    'SibSp' : [0],
    'Parch' : [0],
    'Fare' : [8.05],
    'Embarked' : ['S']
})

print('New Messanger Prediction',
best_model.predict(new_passenger))


Final Accuracy: 0.8044692737430168
New Messanger Prediction [0]


## save

In [10]:
joblib.dump(
    best_model,
    'titanic_survival_model.pkl'
)
print('Model Saved successfully')

Model Saved successfully
